# RAG from scratch: inspect every transformation

This notebook uses the package in `src/`. The CLI and notebook therefore teach the same implementation instead of drifting into two copies.

In [ ]:
from pathlib import Path
from rag_workshop.documents import load_json_documents
from rag_workshop.chunking import chunk_documents
from rag_workshop.embeddings import HashingEmbedder
from rag_workshop.pipeline import RAGPipeline

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
documents = load_json_documents(root / "data" / "knowledge_base.json")
[(document.title, len(document.text.split())) for document in documents]

## 1. Chunking

Overlap repeats a small boundary so a fact is less likely to be cut in half.

In [ ]:
chunks = chunk_documents(documents, chunk_size=45, overlap=10)
[(chunk.id, chunk.start_word, chunk.end_word, chunk.text) for chunk in chunks]

## 2. Embeddings

The local embedder hashes words into a fixed vector. It is useful for learning and deterministic tests, but a semantic model is much stronger in production.

In [ ]:
embedder = HashingEmbedder(dimensions=32)
vectors = embedder.embed([chunk.text for chunk in chunks])
len(vectors), len(vectors[0]), vectors[0][:8]

## 3. Retrieval and prompt construction

The question and stored chunks use the same embedder. We sort cosine scores and place only the best evidence into the prompt.

In [ ]:
pipeline = RAGPipeline(embedder=embedder)
pipeline.index(documents, chunk_size=45, overlap=10)
question = "How many remote days per week are allowed?"
prompt, results = pipeline.prepare(question, top_k=2)
[(round(result.score, 3), result.chunk.source) for result in results]

In [ ]:
print(prompt)

## Challenge

Change `chunk_size`, `overlap`, and `top_k`. Predict the result before running the cell. Then add a difficult question to the dataset and write an evaluation case that checks both the retrieved source and whether the final answer is supported.